# 4. Encoding, scaling, and transformations

## Small idea: representation changes what a model can learn

Categories and numerical measurements require different transformations. The correct
choice depends on meaning, distribution, and the model that will consume the features.

**Learning goals**

- distinguish nominal and ordinal categories;
- encode unseen categories safely;
- compare standard, min–max, and robust scaling;
- apply log and polynomial transformations without using test results to choose them.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (
    FunctionTransformer,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
    RobustScaler,
    StandardScaler,
)

## 1. Nominal categories: one-hot encoding

Nominal values have no meaningful order. Assigning `free=0`, `picture=1`, and
`interview=2` would create a false numerical distance.

In [ ]:
task_train = pd.DataFrame({"task_type": ["free", "picture", "free", "social"]})
task_future = pd.DataFrame({"task_type": ["picture", "interview"]})

one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
task_train_encoded = one_hot.fit_transform(task_train)
task_future_encoded = one_hot.transform(task_future)

print(one_hot.get_feature_names_out().tolist())
print(task_train_encoded)
print("Future rows:\n", task_future_encoded)

The unseen `interview` category becomes all zeros for this feature block. Monitor unknown
categories in production; ignoring them preserves shape but does not make them harmless.
High-cardinality columns may require rare-category grouping, hashing, or a domain-specific
representation instead of thousands of one-hot columns.

## 2. Ordinal categories: encode a real order

Proficiency levels have a documented order. Supply the order explicitly rather than
relying on alphabetical sorting.

In [ ]:
level_train = pd.DataFrame({"proficiency": ["A2", "B1", "B2", "C1"]})
level_future = pd.DataFrame({"proficiency": ["B1", "C2"]})

ordinal = OrdinalEncoder(
    categories=[["A1", "A2", "B1", "B2", "C1"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

ordinal.fit(level_train)
print(ordinal.transform(level_train).ravel())
print("Future:", ordinal.transform(level_future).ravel())

Use an ordinal encoding only when the ordering is substantively meaningful. Even then,
equal numerical gaps between adjacent levels are an assumption, not a linguistic fact.

## 3. Compare scaling strategies

In [ ]:
numeric_train = pd.DataFrame({
    "tokens": [60.0, 80.0, 100.0, 120.0, 900.0],
    "response_seconds": [40.0, 55.0, 70.0, 65.0, 310.0],
})
numeric_future = pd.DataFrame({
    "tokens": [75.0, 1_400.0],
    "response_seconds": [50.0, 420.0],
})

scalers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler(),
}

for name, scaler in scalers.items():
    scaler.fit(numeric_train)
    transformed = scaler.transform(numeric_future)
    print(f"{name}:\n{np.round(transformed, 2)}")

- `StandardScaler` centers by the mean and scales by standard deviation.
- `MinMaxScaler` maps the training range, usually to `[0, 1]`; future values may fall outside it.
- `RobustScaler` uses the median and interquartile range and is less influenced by extremes.

Distance-based, gradient-based, and regularized linear models often benefit from scaling.
Many tree models do not require it. Stage 6 will connect transformations to algorithms.

## 4. Log-transform a skewed positive feature

In [ ]:
log_transformer = FunctionTransformer(
    np.log1p,
    validate=True,
    feature_names_out="one-to-one",
)

token_values = pd.DataFrame({"tokens": [0, 10, 100, 1_000, 10_000]})
token_values["log1p_tokens"] = log_transformer.transform(
    token_values[["tokens"]]
).ravel()
token_values

`log1p(x)` is defined at zero and compresses large positive values. It changes the unit and
interpretation, so record it in the preprocessing specification.

## 5. Polynomial and interaction features

In [ ]:
measurements_train = pd.DataFrame({
    "tokens": [80.0, 120.0, 160.0],
    "sentences": [5.0, 8.0, 9.0],
})
measurements_future = pd.DataFrame({
    "tokens": [100.0],
    "sentences": [6.0],
})

polynomial = PolynomialFeatures(degree=2, include_bias=False)
train_polynomial = polynomial.fit_transform(measurements_train)
future_polynomial = polynomial.transform(measurements_future)

print(polynomial.get_feature_names_out().tolist())
print(train_polynomial)
print("Future:\n", future_polynomial)

Polynomial expansion is a preprocessing transformation; choosing its degree is a model-
selection decision. Do not try many degrees and select the one with the best test score.
Stage 6 will place degree selection inside validation or cross-validation.

## Tiny checkpoint

Choose a representation for task type, ordered proficiency, heavily skewed token count,
and an interaction between token count and sentence count. State which choices learn
parameters from training data.